In [18]:
from ollama import Client
from llama_cpp import Llama
import json
import re
import logging
import sqlparse
from datetime import datetime
# Connexion à Ollama
client = Client(host='http://localhost:11434') 

In [19]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("llama3_trace.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [20]:
# 📊 Schéma simulé
erp_schema = {
    "tables": {
        "commandes": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "date_commande": "DATE",
                "montant": "DECIMAL(10,2)",
                "client_id": "INT FOREIGN KEY",
                "produit_id": "INT FOREIGN KEY"
            },
            "description": "Commandes passées par les clients"
        },
        "clients": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "secteur": "VARCHAR(50)",
                "ville": "VARCHAR(50)"
            },
            "description": "Informations clients"
        },
        "produits": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "categorie": "VARCHAR(50)",
                "prix_unitaire": "DECIMAL(10,2)"
            },
            "description": "Catalogue produits"
        },
        "fournisseurs": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "pays": "VARCHAR(50)",
                "categorie_fournisseur": "VARCHAR(50)"
            },
            "description": "Fournisseurs disponibles"
        }
    },
    "relations": {
        "commandes.client_id": "clients.id",
        "commandes.produit_id": "produits.id"
    }
}


In [21]:
def format_schema_detailed(schema):
    tables_desc = []
    for table, info in schema["tables"].items():
        cols = ", ".join([f"{col} ({type_col})" for col, type_col in info["columns"].items()])
        tables_desc.append(f"Table {table}: {cols} - {info['description']}")
    
    relations = "\nRelations:\n" + "\n".join([f"- {rel}" for rel in schema["relations"]])
    return "\n".join(tables_desc) + relations

In [34]:
def generate_prompt(question: str, schema: dict) -> str:
    return f"""
Tu es un moteur d’analyse linguistique expert intégré à un système ERP. Ta tâche est d’analyser la question ci-dessous et d’en extraire toutes les informations nécessaires à la génération d’une requête SQL. 

QUESTION UTILISATEUR :
"{question}"

SCHÉMA DE LA BASE DE DONNÉES :
{format_schema_detailed(schema)}

OBJECTIF :
Analyse la question et produis une sortie **au format texte structuré strict**, contenant toutes les sections obligatoires, même si certaines sont vides. N’invente jamais de table ou colonne non mentionnée dans le schéma.
et  - N'inclure QUE les colonnes strictement nécessaires pour répondre à la question
   -  Exclure les colonnes techniques (ex: `id`) sauf si explicitement demandées
   -  Faire la meilleur extrcation possible des données
   -  N'ajoute aucun texte explicatif, uniquement les sections demandées
   -  Respecte **strictement** le format suivant pour la sortie, sans ajouter de texte supplémentaire ni explications 
   - Ajoute les filtres et jointures si nécessaire, mais ne les invente pas
    -  Utilise les fonctions d'agrégation si nécessaire, mais ne les invente
pas si elles ne sont pas nécessaires
    -  Utilise les fonctions de date si nécessaire, mais ne les invente pas si elles ne sont pas nécessaires
    - N'inclut jamais des filtres dans la partie colonnes, utilise uniquement la section FILTRES pour les conditions
FORMAT À RESPECTER :

INTENTION: ...
TABLES: [...]
COLONNES: [...]
FILTRES: [...] (si ils existent)
JOINTURES: [...] (si elles existent)
AGRÉGATION: ... (si elle existe)

RÈGLES :
- Ne saute **aucune section**
- Respecte **strictement** les noms du schéma
- Pour les dates : `EXTRACT(MONTH...)`, `EXTRACT(YEAR...)`
- Valeurs textuelles entourées de `'...'`

EXEMPLE :
INTENTION: SELECT  
TABLES: [commandes]  
COLONNES: [commandes.id, commandes.date_commande]  
FILTRES: [EXTRACT(MONTH FROM commandes.date_commande) BETWEEN 1 AND 3, EXTRACT(YEAR FROM commandes.date_commande) = 2024]  
JOINTURES: []  
AGRÉGATION:

Essaie d'utilisaer bcp ton analyse tu as le shéma tu as tous analse bien la réquete et donne les meilleur extrcation et les plus adequats.
""".strip()


In [31]:
def analyser_question_ollama(question):
    prompt = generate_prompt(question, erp_schema)
    response = client.chat(model='llama3.2', messages=[
        {"role": "user", "content": prompt}
    ])
    return response['message']['content']

In [35]:
question_test = "Quels clients ont commandé plus de 3 fois ?"
reponse = analyser_question_ollama(question_test)
print("🧠 Réponse NLP via llama3.2 (Ollama):\n")
print(reponse)

2025-07-07 13:40:20,357 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


🧠 Réponse NLP via llama3.2 (Ollama):

INTENTION: Select des clients qui ont commandé plus de 3 fois

TABLES:
- [commandes]
- [clients]

COLONNES:
- [EXTRACT(MONTH FROM commandes.date_commande)]
- [EXTRACT(YEAR FROM commandes.date_commande)]

FILTRES: [commandes.client_id IN (SELECT id FROM clients)]

JOINTURES:
[]


In [8]:
def parse_response_to_json_multilines(response_text: str) -> dict:
    import re

    result = {
        "intention": "",
        "tables": [],
        "colonnes": [],
        "filtres": [],
        "jointures": [],
        "agregation": ""
    }

    current_section = None
    lines = response_text.strip().splitlines()
    i = 0

    while i < len(lines):
        line = lines[i].strip()
        if not line:
            i += 1
            continue

        if line.startswith("INTENTION:"):
            # Si la ligne contient une suite directe
            content = line[len("INTENTION:"):].strip()
            if content:
                result["intention"] = content
            else:
                # Sinon prendre la ligne suivante si elle existe
                i += 1
                if i < len(lines):
                    next_line = lines[i].strip()
                    result["intention"] = next_line
            current_section = None

        elif line.startswith("TABLES:"):
            current_section = "tables"
            remainder = line[len("TABLES:"):].strip()
            if remainder.startswith("["):
                inner = remainder[1:-1]
                items = [i.strip() for i in inner.split(',') if i.strip()]
                result["tables"].extend(items)

        elif line.startswith("COLONNES:"):
            current_section = "colonnes"
            remainder = line[len("COLONNES:"):].strip()
            if remainder.startswith("["):
                inner = remainder[1:-1]
                items = [i.strip() for i in inner.split(',') if i.strip()]
                result["colonnes"].extend(items)

        elif line.startswith("FILTRES:"):
            current_section = "filtres"
            remainder = line[len("FILTRES:"):].strip()
            if remainder:
                items = [i.strip() for i in re.split(r',(?![^(]*\))', remainder) if i.strip()]
                result["filtres"].extend(items)

        elif line.startswith("JOINTURES:"):
            current_section = "jointures"
            remainder = line[len("JOINTURES:"):].strip()
            if remainder.startswith("["):
                inner = remainder[1:-1]
                items = [i.strip() for i in inner.split(',') if i.strip()]
                result["jointures"].extend(items)

        elif line.startswith("AGRÉGATION:"):
            result["agregation"] = line[len("AGRÉGATION:"):].strip()
            current_section = None

        elif current_section:
            if line.startswith("[") and line.endswith("]"):
                inner = line[1:-1]
                items = [i.strip() for i in re.split(r',(?![^(]*\))', inner) if i.strip()]
                result[current_section].extend(items)
            else:
                items = [i.strip() for i in re.split(r',(?![^(]*\))', line) if i.strip()]
                result[current_section].extend(items)

        i += 1

    return result


In [9]:
# parsed = parse_response_to_json_multilines(reponse)
# print(json.dumps(parsed, indent=2, ensure_ascii=False))

In [11]:
questions = [
    "Combien de produits différents a chaque client commandé ?",
    "Liste les commandes groupées par produit.",
    "Donne-moi la commande avec le montant maximum.",
    "Quels produits ont été commandés par les clients de Rabat ?",
    "Quelles sont les catégories les plus commandées ?",
    "Quels clients ont commandé plus de 3 fois ?",
    "Quelles commandes concernent des produits à plus de 500 MAD ?",
    "Liste des clients ayant commandé en mars 2023.",
    "Combien de commandes ont été passées par secteur ?"
]

for idx, q in enumerate(questions, 1):
    try:
        logger.info(f"📤 Question {idx}: {q}")
        reponse = analyser_question_ollama(q)
        
        logger.debug(f"🧠 Réponse brute Mistral:\n{reponse}")
        logger.info(f"🧠 Réponse NLP reçue (question {idx}):\n{reponse}")

        parsed = parse_response_to_json_multilines(reponse)
        
        logger.info(f"✅ Question {idx} analysée avec succès.")
        logger.debug(f"📦 JSON structuré (question {idx}):\n{json.dumps(parsed, indent=2, ensure_ascii=False)}")

        print(f"\n🔢 Question {idx}")
        print(f"❓ {q}")
        print(f"📦 Réponse Mistral :\n{json.dumps(parsed, indent=2, ensure_ascii=False)}")

    except Exception as e:
        logger.error(f"❌ Erreur pour la question {idx} : {q} - {str(e)}")
        print(f"\n❌ Erreur pour la question {idx} : {q}")
        print(e)



2025-07-07 11:57:25,599 - INFO - 📤 Question 1: Combien de produits différents a chaque client commandé ?
2025-07-07 11:58:21,832 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2025-07-07 11:58:21,857 - INFO - 🧠 Réponse NLP reçue (question 1):
INTENTION:
Trouver le nombre de produits différents commandés par chaque client.

TABLES:
[commandes]
[clients]

COLONNES:
[commandes.id, commandes.date_commande, clients.nom, commandes.produit_id]
[clients.id, clients.secteur, clients.ville]

FILTRES: 
[]

JOINTURES:
[commandes.client_id = clients.id]

AGRÉGATION:
Utiliser GROUP BY pour regrouper les produits par client et compter le nombre de produits différents pour chaque client.
2025-07-07 11:58:21,857 - INFO - ✅ Question 1 analysée avec succès.
2025-07-07 11:58:21,865 - INFO - 📤 Question 2: Liste les commandes groupées par produit.



🔢 Question 1
❓ Combien de produits différents a chaque client commandé ?
📦 Réponse Mistral :
{
  "intention": "Trouver le nombre de produits différents commandés par chaque client.",
  "tables": [
    "commandes",
    "clients"
  ],
  "colonnes": [
    "commandes.id",
    "commandes.date_commande",
    "clients.nom",
    "commandes.produit_id",
    "clients.id",
    "clients.secteur",
    "clients.ville"
  ],
  "filtres": [],
  "jointures": [
    "commandes.client_id = clients.id"
  ],
  "agregation": ""
}


2025-07-07 11:58:49,372 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2025-07-07 11:58:49,375 - INFO - 🧠 Réponse NLP reçue (question 2):
INTENTION: Listes des commandes groupées par produit.

TABLES:
[commandes]
[produits]

COLONNES:
[commandes.id, 
commandes.date_commande, 
commandes.montant, 
commandes.produit_id]  
[produits.id, 
produits.nom, 
produits.categorie, 
produits.prix_unitaire]  

FILTRES: [] (pas de filtres spécifiques)

JOINTURES:
[]
(une jointure est nécessaire pour lier les commandes et produits)
[commandes.produit_id = produces.id]

AGRÉGATION:
GROUP BY
  [commandes.produit_id]
2025-07-07 11:58:49,379 - INFO - ✅ Question 2 analysée avec succès.
2025-07-07 11:58:49,379 - INFO - 📤 Question 3: Donne-moi la commande avec le montant maximum.



🔢 Question 2
❓ Liste les commandes groupées par produit.
📦 Réponse Mistral :
{
  "intention": "Listes des commandes groupées par produit.",
  "tables": [
    "commandes",
    "produits"
  ],
  "colonnes": [
    "[commandes.id",
    "commandes.date_commande",
    "commandes.montant",
    "commandes.produit_id]",
    "[produits.id",
    "produits.nom",
    "produits.categorie",
    "produits.prix_unitaire]"
  ],
  "filtres": [
    "[] (pas de filtres spécifiques)"
  ],
  "jointures": [
    "(une jointure est nécessaire pour lier les commandes et produits)",
    "commandes.produit_id = produces.id"
  ],
  "agregation": ""
}


2025-07-07 11:59:13,117 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2025-07-07 11:59:13,119 - INFO - 🧠 Réponse NLP reçue (question 3):
INTENTION: Retourner la commande avec le montant maximum pour chaque client.

TABLES:
[commandes]
[clients]

COLONNES:
[commandes.id, commandes.date_commande, commandes.montant, commandes.client_id, clients.nom, clients.secteur, clients.ville]
[clients.id, clients.nom, clients.secteur, clients.ville]

FILTRES: 
EXTRACT(MONTH FROM commandes.date_commande) BETWEEN 1 AND 12 

JOINTURES:
[commandes.client_id = clients.id]
2025-07-07 11:59:13,121 - INFO - ✅ Question 3 analysée avec succès.
2025-07-07 11:59:13,125 - INFO - 📤 Question 4: Quels produits ont été commandés par les clients de Rabat ?


...


🔢 Question 7
❓ Quelles commandes concernent des produits à plus de 500 MAD ?
📦 Réponse Mistral :
{
  "intention": "Extraire les informations nécessaires pour générer une requête SQL pour répondre à la question \"Quelles commandes concernent des produits à plus de 500 MAD ?\"",
  "tables": [
    "- commandes",
    "- produits"
  ],
  "colonnes": [
    "- commandes.id",
    "- commandes.date_commande",
    "- commandes.montant",
    "- produits.id",
    "- produits.prix_unitaire",
    "- commandes.produit_id"
  ],
  "filtres": [
    "[commandes.montant > 500]"
  ],
  "jointures": [
    "- commandes INNER JOIN produits ON commandes.produit_id = produits.id"
  ],
  "agregation": "non requise"
}


2025-07-07 12:01:59,398 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2025-07-07 12:01:59,402 - INFO - 🧠 Réponse NLP reçue (question 8):
**INTENTION**
Liste des clients ayant commandé en mars 2023.

**TABLES**
[commandes, clients]

**COLONNES**
[commandes.id, commandes.date_commande, clients.nom, clients.secteur, clients.ville]

**FILTRES**
[EXTRACT(MONTH FROM commandes.date_commande) = 3, EXTRACT(YEAR FROM commandes.date_commande) = 2023]

**JOINTURES**
[commandes.client_id = clients.id]

**AGRÉGATION**
N/A

Pour générer la requête SQL, nous utiliserons les informations suivantes :

* Les utilisateurs ayant commandé en mars 2023 doivent appartenir à la table "commandes".
* La date de commande est filtrée pour correspondre au mois mars et à l'année 2023.
* La clé étrangère "client_id" relie les commandes aux clients.

La requête SQL serait donc :

```sql
SELECT 
    c.id, c.date_commande, c.client_id,
    cl.nom AS nom_client, cl.secteur, cl.ville
FROM 



🔢 Question 8
❓ Liste des clients ayant commandé en mars 2023.
📦 Réponse Mistral :
{
  "intention": "",
  "tables": [],
  "colonnes": [],
  "filtres": [],
  "jointures": [],
  "agregation": ""
}


2025-07-07 12:02:22,813 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2025-07-07 12:02:22,816 - INFO - 🧠 Réponse NLP reçue (question 9):
INTENTION:
Rechercher le nombre de commandes passées par secteur.

TABLES:
[commandes, clients]

COLONNES:
[commandes.id, commandes.date_commande, commandes.montant, commandes.client_id, commandes.produit_id,
clients.id, clients.nom, clients.secteur]

FILTRES: 
[clients.secteur IS NOT NULL]  -> 1ère condition

JOINTURES: [clients.id = commandes.client_id] 

AGRÉGATION:
GRUPPER BY 'clients.secteur'
2025-07-07 12:02:22,818 - INFO - ✅ Question 9 analysée avec succès.



🔢 Question 9
❓ Combien de commandes ont été passées par secteur ?
📦 Réponse Mistral :
{
  "intention": "Rechercher le nombre de commandes passées par secteur.",
  "tables": [
    "commandes",
    "clients"
  ],
  "colonnes": [
    "[commandes.id",
    "commandes.date_commande",
    "commandes.montant",
    "commandes.client_id",
    "commandes.produit_id",
    "clients.id",
    "clients.nom",
    "clients.secteur]"
  ],
  "filtres": [
    "[clients.secteur IS NOT NULL]  -> 1ère condition"
  ],
  "jointures": [
    "clients.id = commandes.client_id"
  ],
  "agregation": ""
}
